# Pemisahan Foto Formulir Tulisan Tangan Aksara Sunda

Memotong foto lembar formulir (tabel 8 kolom x 10 baris = 80 aksara) menjadi satu citra per aksara (224x224 px).

- Input : `foto_formulir/<kelas>/<foto>`
- Output: `dataset_tulisan_tangan/<kelas>/<kelas>_<nnn>.jpg`

Logika pemotongan ada di `split_dataset_manual.py` (satu sumber, dipakai juga lewat terminal).

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

import split_dataset_manual as sdm

INPUT_DIR = Path("../foto_formulir")
OUTPUT_DIR = Path("../dataset_tulisan_tangan")
print("Input :", INPUT_DIR.resolve())
print("Output:", OUTPUT_DIR.resolve())

## 1. Cek satu lembar
Tampilkan tabel setelah ditegakkan dan diluruskan, beserta batas kotak yang terdeteksi.

In [ ]:
contoh = sorted(p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in sdm.EXTS)[0]
warped = sdm.warp_table(sdm.orient_upright(cv2.imread(str(contoh))))
boxes, _, _ = sdm.cell_boxes(warped)

vis = warped.copy()
for top, bot, left, right in boxes.values():
    cv2.rectangle(vis, (left, top), (right, bot), (0, 0, 255), 2)

plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(str(contoh.relative_to(INPUT_DIR)))
plt.axis("off")
plt.show()

## 2. Proses semua lembar
Folder output lama dihapus dulu agar penomoran tidak dobel.

In [ ]:
sdm.process_dataset(INPUT_DIR, OUTPUT_DIR, clean=True)

## 3. Preview hasil per kelas

In [ ]:
N = 10
kelas = sorted(d for d in OUTPUT_DIR.iterdir() if d.is_dir())
fig, axs = plt.subplots(len(kelas), N, figsize=(N * 1.3, len(kelas) * 1.4), squeeze=False)
for row, d in zip(axs, kelas):
    files = sorted(d.glob("*.jpg"))[:N]
    for ax in row:
        ax.axis("off")
    for ax, f in zip(row, files):
        ax.imshow(cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB))
        ax.set_title(f.stem, fontsize=7)
plt.tight_layout()
plt.show()